# Built-in Tools

Every `AmphibiousAutoma` agent already has seven tools wired in — a baseline capability surface that lets the LLM (or your workflow) talk to a shell, the filesystem, and the human operator without any per-project setup. In this tutorial we'll meet them, use them in both **Agent mode** and **Workflow mode**, walk through the read-before-modify safety model, and learn how to scope which tools an agent can reach.

## Practical Scenario

We'll build a small **"config patcher"** that operates on a temporary directory of source files. The agent will:

1. Find Python files via `glob`,
2. Search them for a string with `grep`,
3. Read a specific file via `read_file`,
4. Apply a fix with `edit_file`, and
5. Verify the fix with `bash`.

All of this happens with **zero custom tools** — every operation comes from the framework's built-in tool surface.

## The Seven Built-in Tools

Every `AmphibiousAutoma.arun()` call auto-injects these into `context.tools` (subject to filtering, which we'll cover later):

| Tool | What it does |
|------|--------------|
| `request_human` | Pause and ask the human operator a question (HITL) |
| `bash` | Execute a shell command (stdout / stderr / exit_code captured) |
| `read_file` | Read a file with line numbers; required before `write_file` / `edit_file` modify it |
| `write_file` | Create or overwrite a file (read-before-overwrite enforced) |
| `edit_file` | Exact-string replacement with uniqueness check; `replace_all` for refactors |
| `glob` | Find files by pattern, sorted by mtime |
| `grep` | Regex content search across files |

Names are snake_case. They work in **Agent mode** (the LLM picks them on its own) and **Workflow mode** (`yield ActionCall("tool_name", ...)`). Tools raise on validation failures — the framework converts the exception into `ActionStepResult(success=False)` so the LLM sees the error and adapts.

> `request_human` is the same auto-injected tool covered by the [Human-in-the-Loop tutorial](../core_mechanism/human_in_the_loop.ipynb); we won't re-demonstrate it here.

## Setting Up a Sandbox

Rather than touch your real filesystem, let's prepare a temporary directory with two small Python files. The agent will operate exclusively inside this sandbox.

In [18]:
import tempfile
from pathlib import Path

sandbox = Path(tempfile.mkdtemp(prefix="amphibious-tutorial-"))

(sandbox / "config.py").write_text(
    "DEBUG = False\n"
    "TIMEOUT = 30\n"
    "RETRIES = 3\n"
)

(sandbox / "service.py").write_text(
    "def handle_request(request):\n"
    "    if DEBUG:\n"
    "        log(request)\n"
    "    return process(request)\n"
)

print(f"Sandbox: {sandbox}")
print(f"Files:   {[p.name for p in sandbox.iterdir()]}")

Sandbox: /var/folders/pg/wph0nst97ks8x17hw6wwcz200000gn/T/amphibious-tutorial-47y4y_v8
Files:   ['service.py', 'config.py']


## Initialize the LLM

We'll use Agent mode for one example, so we need an LLM. Pure Workflow mode does not require one.

In [ ]:
import os

from bridgic.llms.openai import OpenAILlm, OpenAIConfiguration

llm = OpenAILlm(
    api_key=os.environ.get("API_KEY"),
    api_base=os.environ.get("BASE_URL"),
    timeout=30,
    configuration=OpenAIConfiguration(
        model=os.environ.get("MODEL_NAME"),
        temperature=0.0,
        max_tokens=16384,
    ),
)

## Agent Mode — Let the LLM Pick the Tools

Notice that we pass nothing in `tools=[...]`: the agent already has all seven built-ins. The LLM decides which to call to satisfy the goal.

In [10]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit


class CodeInvestigator(AmphibiousAutoma[CognitiveContext]):
    investigator = think_unit(
        CognitiveWorker.inline(
            "Investigate the directory the user describes. "
            "List the Python files, find any place that uses 'DEBUG', "
            "and report what you find. Response in JSON format."
        ),
        max_attempts=10,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.investigator


agent = CodeInvestigator(llm=llm, verbose=True)
report = await agent.arun(
    goal=f"Investigate the directory at {sandbox}. Use glob to list .py files and grep to find DEBUG.",
)
print(report)

[19:41:10.639] [Router] (_amphibious_automa.py:1615) Ferrying to AGENT mode
[19:41:10.640] [Observe] (_amphibious_automa.py:897) _PromptWorker: None
[19:41:17.951] [Think] (_amphibious_automa.py:903) _PromptWorker: finish=False, step=I'll investigate the specified directory by listing all Python files and searching for occurrences of 'DEBUG' in them. Using glob to find .py files and grep to search for DEBUG patterns will give us the information needed to report on what we find.
[19:41:17.954] [Act] (_amphibious_automa.py:909) _PromptWorker:
{
    "content": "I'll investigate the specified directory by listing all Python files and searching for occurrences of 'DEBUG' in them. Using glob to find .py files and grep to search for DEBUG patterns will give us the information needed to report on what we find.",
    "result": {
        "results": [
            {
                "tool_id": "call_0",
                "tool_name": "glob",
                "tool_arguments": {
                    "pa

Watch the verbose log: the LLM autonomously calls `glob` first to enumerate files, then `grep` to locate the `DEBUG` references, and produces a summary. None of these tools were declared in `tools=[...]` — they came for free.

## Workflow Mode — Calling Built-ins Deterministically

In `on_workflow`, **you** choose the order. `yield ActionCall("tool_name", ...)` works the same for built-ins as for any user-defined tool. The example below stitches `glob → read_file → edit_file → bash` into a deterministic patch pipeline.

In [19]:
from bridgic.amphibious import ActionCall


class ConfigPatcher(AmphibiousAutoma[CognitiveContext]):
    def __init__(self, target_dir: Path, **kwargs):
        super().__init__(**kwargs)
        self.target_dir = target_dir

    async def on_workflow(self, ctx: CognitiveContext):
        target = self.target_dir / "config.py"

        # 1. Discover files in the directory.
        yield ActionCall("glob", pattern="*.py", path=str(self.target_dir))

        # 2. Read the target file. This is REQUIRED before edit_file may touch it.
        yield ActionCall("read_file", file_path=str(target))

        # 3. Apply the fix.
        yield ActionCall(
            "edit_file",
            file_path=str(target),
            old_string="DEBUG = False",
            new_string="DEBUG = True",
        )

        # 4. Verify with a shell command.
        verify = yield ActionCall("bash", command=f"grep '^DEBUG' {target}")
        self.set_final_answer(f"Patched. Verification:\n{verify[0].result}")


workflow = ConfigPatcher(target_dir=sandbox)  # No LLM needed for pure workflow.
result = await workflow.arun(goal="Flip DEBUG to True")
print(result)

Patched. Verification:
<stdout>
DEBUG = True

</stdout>
<exit_code>0</exit_code>


## The Read-Before-Modify Invariant

`write_file` (for existing files) and `edit_file` refuse to act on a path unless `read_file` was called on it first **in this `arun()` run**, AND the file hasn't changed externally since that read. The tracker is reset at every `arun()` entry, scoped to a single run.

This is a deliberate safety net: it keeps the LLM from blindly clobbering content it has never seen, and detects races where another process modified the file between the read and the write.

Watch what happens if we skip the `read_file` step:

In [13]:
class UnsafeEdit(AmphibiousAutoma[CognitiveContext]):
    def __init__(self, target: Path, **kwargs):
        super().__init__(**kwargs)
        self.target = target

    async def on_workflow(self, ctx: CognitiveContext):
        # No read_file beforehand — this will fail.
        yield ActionCall(
            "edit_file",
            description="apply patch without reading first",
            file_path=str(self.target),
            old_string="TIMEOUT = 30",
            new_string="TIMEOUT = 60",
        )


unsafe = UnsafeEdit(target=sandbox / "config.py")

try:
    await unsafe.arun(goal="patch without reading")
except RuntimeError as e:
    print("Workflow failed as expected:")
    print(str(e))

Workflow failed as expected:
Tool execution failed for: apply patch without reading first — edit_file: You must use the read_file tool to read /var/folders/pg/wph0nst97ks8x17hw6wwcz200000gn/T/amphibious-tutorial-r_s75d0j/config.py at least once in the conversation before modifying it.


Note how the error chain travels:

1. `edit_file` raises `RuntimeError("You must use the read_file tool to read ... at least once ...")`.
2. The framework's per-tool exception handler in `_action_tool_call._run_one` catches it and produces an `ActionStepResult(success=False, error=...)`.
3. `_run_workflow` aggregates the failed result into a `RuntimeError("Tool execution failed for: ... — edit_file: ...")`, which propagates out of `arun()` (because pure WORKFLOW mode does not fall back).

In Agent mode the same error becomes part of the next observation, so the LLM sees the message and typically corrects itself by calling `read_file` first.

## Filtering Built-in Tools

Sometimes you want to constrain what an agent can reach — e.g. give a code reviewer agent read-only access, or strip `bash` from a sensitive workflow. Two knobs control which tools are injected:

- **Class-level**: a `builtin_tools` class attribute, declared as a `frozenset` of tool names.
- **Per-run**: an `arun(builtin_tools=[...])` keyword argument that overrides the class attribute.

Either knob set to an empty iterable opts out entirely. `None` (the default for both) injects all seven. Unknown names raise `ValueError` at `arun()` entry — typos surface immediately rather than producing a silently broken agent.

In [14]:
class ReadOnlyAuditor(AmphibiousAutoma[CognitiveContext]):
    # Only these four — no bash, no write_file, no edit_file.
    builtin_tools = frozenset({"request_human", "read_file", "glob", "grep"})

    auditor = think_unit(
        CognitiveWorker.inline(
            "Audit the code for any use of insecure patterns. Do NOT modify files."
        ),
        max_attempts=10,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.auditor


# Typos fail loudly at arun() entry — try a misspelt name:
auditor = ReadOnlyAuditor(llm=llm)
try:
    await auditor.arun(goal="audit", builtin_tools=["read_files"])  # extra 's'
except ValueError as e:
    print(f"Caught typo at arun() entry:\n{e}")

Caught typo at arun() entry:
Unknown built-in tool name(s) in builtin_tools filter: ['read_files']. Valid names: ['bash', 'edit_file', 'glob', 'grep', 'read_file', 'request_human', 'write_file'].


Two more useful patterns (commented out so this notebook stays cheap to run):

```python
# Per-run override — keep only request_human for this specific call:
await auditor.arun(goal="...", builtin_tools=["request_human"])

# Empty list opts out completely (no built-ins injected at all):
await auditor.arun(goal="...", builtin_tools=[])
```

User-supplied tools (passed via `tools=[...]`) take precedence over built-ins of the same name — the colliding built-in is silently skipped, so you can drop in a custom replacement (e.g. a sandboxed `bash`) without disabling anything.

## Combining with `think_unit` Tool Filters

`think_unit(tools=[...])` further filters what a *particular* think unit can see — even if the agent has all seven built-ins available, a phase can be locked down to a subset. Built-in names work the same as any user-defined tool name. This is the natural way to gate an agent's capabilities by phase.

In [15]:
class PhaseGated(AmphibiousAutoma[CognitiveContext]):
    investigate = think_unit(
        CognitiveWorker.inline("Investigate the codebase. Read-only."),
        tools=["read_file", "glob", "grep"],   # exploration only
        max_attempts=8,
    )
    apply = think_unit(
        CognitiveWorker.inline("Apply the planned fix."),
        tools=["read_file", "edit_file"],       # no bash, no overwrite
        max_attempts=4,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.investigate
        await self.apply

## Cleanup

Remove the temporary sandbox we created at the start.

In [16]:
import shutil

shutil.rmtree(sandbox)
print(f"Removed {sandbox}")

Removed /var/folders/pg/wph0nst97ks8x17hw6wwcz200000gn/T/amphibious-tutorial-r_s75d0j


<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

- **Seven built-in tools** are auto-injected into every `AmphibiousAutoma` agent — `request_human`, `bash`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`. No wiring required.
- They work the same in **Agent mode** (the LLM picks them) and **Workflow mode** (`yield ActionCall("name", ...)`). Tool names are snake_case and identical across both modes.
- **Read-before-modify**: `edit_file` and `write_file` (for existing files) refuse to act unless `read_file` was called first AND the file hasn't changed since. The tracker is per-run, reset at every `arun()` entry.
- **Errors raise**: tools throw on validation failures; the framework converts the exception into `ActionStepResult(success=False)` so the LLM sees the error and adapts. In WORKFLOW mode the failure surfaces as a `RuntimeError`.
- **Filtering** has two layers — `builtin_tools` class attribute (default scope for a subclass) and `arun(builtin_tools=...)` kwarg (per-run override). Unknown names raise `ValueError` at `arun()` entry.
- **`think_unit(tools=[...])`** further scopes what a single phase can reach — useful for splitting an agent into "investigate" and "apply" phases with different capability surfaces.

Next, we'll explore RunMode — the four execution modes and how Amphiflow lets workflows fall back to agent mode when things go off-script.